In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI()

response = client.responses.create(
    model="qwen3.5-flash",
    input="你是谁？"
)
print(response.output_text)

你好！我是 **Qwen3.5**，阿里巴巴通义实验室研发的超大规模语言模型。我能够回答问题、创作文字、分析信息、辅助编程，也能进行多轮对话或处理复杂任务。如果你有任何问题或需要帮助，随时告诉我！


In [3]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="qwen3.5-flash", temperature=0.7, max_tokens=1000)

messages = [
    SystemMessage("你是一名清晰、耐心的 AI 导师。"),
    HumanMessage("一句话解释 什么是 Chat Template。")
]

response = llm.invoke(messages)
print(response.content)

Chat Template 是一种标准化的对话格式规范，用于将系统指令、用户输入和模型回复按特定结构组织起来，确保大模型能准确理解上下文并生成符合预期的回复。


## ReAct Agent：Thought → Action → Observation 循环

ReAct（**Re**ason + **Act**）是最经典的 Agent 范式：

1. **Thought**：模型先"想一想"——我现在需要什么信息？
2. **Action**：选择并调用一个工具（搜索、计算、查数据库……）
3. **Observation**：观察工具返回的结果
4. 回到 1，直到能直接回答用户问题为止

### 经典 LangChain 0.x 写法（参考用）

```python
from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub

prompt = hub.pull("hwchase17/react")          # 预定义的 ReAct Prompt 模板
agent = create_react_agent(llm, tools, prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
executor.invoke({"input": "谁是现任美国总统？他的年龄是多少？"})
```

### LangChain 1.x 的现代写法（本 notebook 使用）

v1 把 Agent 底层迁到了 LangGraph，API 更简洁：

- `langchain.agents.create_agent(model, tools, system_prompt=...)` 直接返回编译好的 Agent 图
- 不再需要手写 ReAct Prompt 模板——Agent 直接走模型原生的 **tool calling** 协议
- 依然保留 Thought-Action-Observation 语义，只是 Thought/Action 合并进 `AIMessage.tool_calls`，Observation 变成 `ToolMessage`

下面用一个"查现任美国总统 + 算他年龄"的经典问题来跑通完整循环。

In [1]:
from datetime import date

from langchain_core.tools import tool


@tool
def search_web(query: str) -> str:
    """联网搜索（演示版）。输入一段查询文本，返回检索到的摘要信息。"""
    q = query.lower()
    if "president" in q or "总统" in q:
        # 刻意只给出生日期，不给当前年份，迫使 Agent 再调用 get_today + calculate
        return "现任美国总统是 Donald John Trump，他出生于 1946 年 6 月 14 日。"
    if "总理" in q or "premier" in q:
        return "李强，中华人民共和国国务院总理（自 2023 年 3 月起）。"
    return f"（搜索）未找到与 '{query}' 高度相关的结果，建议换个关键词再试。"


@tool
def get_today() -> str:
    """获取今天的日期，ISO 格式 YYYY-MM-DD。计算"年龄/距今多久"之前必须先调用它拿到当前日期。"""
    return date.today().isoformat()


@tool
def calculate(expression: str) -> str:
    """执行基础数学计算。输入一个 Python 可求值的算术表达式，例如 '2025 - 1946'。"""
    try:
        result = eval(expression, {"__builtins__": {}}, {})
    except Exception as exc:
        return f"计算出错：{exc}"
    return str(result)


tools = [search_web, get_today, calculate]
[t.name for t in tools]

['search_web', 'get_today', 'calculate']

### 构建并运行 Agent

下面用 `create_agent` 组装 Agent，然后用 `stream(..., stream_mode="values")` 把每一步的最新消息打印出来——你会清楚地看到：

- `AIMessage` 里带 `tool_calls` = Thought + Action
- `ToolMessage` = Observation
- 最后一条不带 `tool_calls` 的 `AIMessage` = Final Answer

In [4]:
from langchain.agents import create_agent

system_prompt = (
    "你是一个严格使用工具的智能助手。硬性规则：\n"
    "1. 涉及实时事实（人物、职位、时事）必须用 search_web；\n"
    "2. 涉及'现在/今天/当前年份'必须用 get_today，不要凭直觉猜年份；\n"
    "3. 涉及任何算术（年龄、年份差、数字运算）必须用 calculate，不要心算；\n"
    "4. 按 Thought → Action → Observation 的顺序一步一步来，最后用简洁的中文回答。"
)

agent = create_agent(llm, tools, system_prompt=system_prompt)

question = "谁是现任美国总统？他今天的年龄是多少？"

printed = 0
for event in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
):
    messages = event["messages"]
    # 只打印新出现的消息，避免重复
    for msg in messages[printed:]:
        msg.pretty_print()
    printed = len(messages)

================================ Human Message =================================

谁是现任美国总统？他今天的年龄是多少？
================================== Ai Message ==================================
Tool Calls:
  search_web (call_2bf758abaec14ee8b23e70a2)
 Call ID: call_2bf758abaec14ee8b23e70a2
  Args:
    query: 现任美国总统是谁 2025
  get_today (call_f182282867834b6599800e47)
 Call ID: call_f182282867834b6599800e47
  Args:
================================= Tool Message =================================
Name: search_web

现任美国总统是 Donald John Trump，他出生于 1946 年 6 月 14 日。
================================= Tool Message =================================
Name: get_today

2026-04-18
================================== Ai Message ==================================
Tool Calls:
  calculate (call_5c736a1fa71e49e79671089c)
 Call ID: call_5c736a1fa71e49e79671089c
  Args:
    expression: 2026 - 1946
================================= Tool Message =================================
Name: calculate

80
=========================